# Stage 1: CPI Inflation and MPC Decisions Data Preparation
This notebook cleans and prepares the monthly Consumer Price Index (CPI) dataset and the Reserve Bank of India (RBI) Monetary Policy Committee (MPC) decisions dataset. It merges them using a backward search so that each MPC decision is aligned with the most recently available CPI data at the time of the meeting.

In [ ]:
import pandas as pd
import numpy as np
import os

## 1. Load and Clean CPI Data
We read the raw CPI data, rename columns, compute Core CPI (General - Food & Fuel), and calculate Year-over-Year (YoY) inflation rates.

In [ ]:
# Load raw CPI excel file
df_cpi_raw = pd.read_excel('../raw_data/CPI-Base-2012.xlsx')

# Clean empty rows
df_cpi_raw = df_cpi_raw.dropna(how='all').reset_index(drop=True)

# Pivot and extract key indexes from row 2 (which contains the header info)
# Columns are: Unnamed: 1 (Month), Unnamed: 2 (Commodity Description), Unnamed: 8 (Combined Index)
# Let's filter for relevant commodities: General Index, Food and beverages, Fuel and light
df_filtered = df_cpi_raw.iloc[4:].copy()
df_filtered = df_filtered.rename(columns={
    'Unnamed: 1': 'month_str',
    'Unnamed: 2': 'commodity',
    'Unnamed: 8': 'value'
})

df_filtered = df_filtered[df_filtered['month_str'].notna() & df_filtered['commodity'].notna()]
df_filtered['value'] = pd.to_numeric(df_filtered['value'], errors='coerce')

# Extract General, Food, and Fuel
general_cpi = df_filtered[df_filtered['commodity'] == 'A) General Index'][['month_str', 'value']].rename(columns={'value': 'cpi_general'})
food_cpi = df_filtered[df_filtered['commodity'] == 'A.1) Food and beverages'][['month_str', 'value']].rename(columns={'value': 'cpi_food'})
fuel_cpi = df_filtered[df_filtered['commodity'] == 'A.5) Fuel and light'][['month_str', 'value']].rename(columns={'value': 'cpi_fuel'})

# Merge the components together
df_cpi = pd.merge(general_cpi, food_cpi, on='month_str')
df_cpi = pd.merge(df_cpi, fuel_cpi, on='month_str')

# Convert month string to datetime
df_cpi['date'] = pd.to_datetime(df_cpi['month_str'], format='%b-%Y', errors='coerce')
df_cpi = df_cpi.dropna(subset=['date']).sort_values('date').reset_index(drop=True)

# Compute Core CPI (General CPI excluding Food (45.63% weight) and Fuel (6.66% weight))
df_cpi['cpi_core'] = df_cpi['cpi_general'] - (df_cpi['cpi_food'] * 0.4563 + df_cpi['cpi_fuel'] * 0.0666)

# Compute YoY Inflation rates
for col in ['cpi_general', 'cpi_food', 'cpi_fuel', 'cpi_core']:
    df_cpi[f'{col}_yoy'] = df_cpi[col].pct_change(12) * 100

# Save cleaned CPI
os.makedirs('../data', exist_ok=True)
df_cpi.to_csv('../data/cleaned_cpi.csv', index=False)
print("Cleaned CPI data saved. Rows:", len(df_cpi))
df_cpi.tail()

## 2. Load and Clean MPC Decisions
We read the voting decisions Excel file, clean footer comments, parse the decisions, and categorize the action (hike, cut, hold).

In [ ]:
# Load raw MPC Excel
df_mpc_raw = pd.read_excel('../raw_data/mpc_decisions.xlsx')
df_mpc_raw = df_mpc_raw.dropna(how='all').reset_index(drop=True)

# Columns are in row 2
df_mpc_raw.columns = df_mpc_raw.iloc[2]
df_mpc_raw = df_mpc_raw.iloc[3:].reset_index(drop=True)

# Extract key columns by positional index
df_mpc = pd.DataFrame({
    'date': df_mpc_raw.iloc[:, 1],
    'repo_rate': df_mpc_raw.iloc[:, 2],
    'raw_decision': df_mpc_raw.iloc[:, 3]
})

# Parse dates and filter footnotes
df_mpc['date'] = pd.to_datetime(df_mpc['date'], errors='coerce')
df_mpc = df_mpc.dropna(subset=['date']).sort_values('date').reset_index(drop=True)

# Clean voting decisions
def clean_bps(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip()
    if val_str in ['P', '', 'nan']:
        return 0.0
    if '(+)' in val_str:
        return float(val_str.replace('(+)', '').strip())
    elif '(-)' in val_str:
        return -float(val_str.replace('(-)', '').strip())
    if val_str.startswith('+'):
        return float(val_str[1:])
    elif val_str.startswith('-'):
        return float(val_str)
    try:
        return float(val_str)
    except ValueError:
        return 0.0

df_mpc['bps_change'] = df_mpc['raw_decision'].apply(clean_bps)
df_mpc['repo_rate'] = pd.to_numeric(df_mpc['repo_rate'], errors='coerce')

# Label is_hike and decision type
df_mpc['is_hike'] = (df_mpc['bps_change'] > 0).astype(int)
df_mpc['decision'] = 'hold'
df_mpc.loc[df_mpc['bps_change'] > 0, 'decision'] = 'hike'
df_mpc.loc[df_mpc['bps_change'] < 0, 'decision'] = 'cut'

print("MPC decisions parsed. Decision distribution:")
print(df_mpc['decision'].value_counts())
df_mpc.tail(10)

## 3. Merge Datasets with Lag Features
We perform a merge-as-of to get the latest monthly CPI data preceding each MPC meeting, then calculate lags and averages.

In [ ]:
# Merge CPI and MPC
merged = pd.merge_asof(
    df_mpc,
    df_cpi[['date', 'cpi_general_yoy', 'cpi_food_yoy', 'cpi_fuel_yoy', 'cpi_core_yoy']],
    on='date',
    direction='backward'
)

# Calculate lag variables and rolling averages
merged['core_lag1'] = merged['cpi_core_yoy'].shift(1)
merged['core_lag2'] = merged['cpi_core_yoy'].shift(2)
merged['core_3m_avg'] = merged['cpi_core_yoy'].rolling(3).mean()

# Save processed merged data
merged.to_csv('../data/processed_cpi_mpc.csv', index=False)
print("Processed merged dataset saved. Rows:", len(merged))
merged.head(10)